# Bank Customer Churn: Exploratory Data Analysis

This notebook explores the supplied bank-customer dataset before modeling. It reports descriptive patterns only; associations do not establish causes of churn.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

DATA_PATH = Path('../data/European_Bank.csv') if Path('../data/European_Bank.csv').exists() else Path('data/European_Bank.csv')
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.astype(str).str.strip()
df.head()

## Dataset and data quality

Check row counts, duplicates, missing values, target balance, and constant fields before interpreting results.

In [ ]:
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Exact duplicate rows: {df.duplicated().sum():,}')
display(df.isna().sum().rename('Missing values').to_frame())
display(df.nunique(dropna=False).rename('Distinct values').to_frame())
display(df['Exited'].value_counts(dropna=False).rename('Customers').to_frame())
print(f'Overall churn rate: {df.Exited.mean():.1%}')

## Customer outcome and age patterns

In [ ]:
df['Outcome'] = df['Exited'].map({0: 'Retained', 1: 'Churned'})
fig = px.histogram(df, x='Age', color='Outcome', marginal='box', barmode='overlay', opacity=.7, title='Age distribution by churn outcome')
fig.show()

## Churn rates by customer group

Rates for small groups can be unstable. Always show sample counts with subgroup rates.

In [ ]:
def churn_summary(data, group):
    return (data.groupby(group, observed=True)['Exited']
            .agg(['mean', 'size']).reset_index()
            .rename(columns={'mean': 'Churn rate', 'size': 'Customers'}))

for column in ['Geography', 'Gender', 'IsActiveMember', 'NumOfProducts']:
    if column in df.columns:
        summary = churn_summary(df, column)
        fig = px.bar(summary, x=column, y='Churn rate', color='Customers',
                     hover_data={'Customers': True, 'Churn rate': ':.1%'},
                     title=f'Churn rate by {column}', color_continuous_scale='Blues')
        fig.update_yaxes(tickformat='.0%')
        fig.show()

## Numeric feature relationships

The scatterplot shows a sample for readability. Correlations describe linear association and are not causal effects.

In [ ]:
sample = df.sample(min(3000, len(df)), random_state=42)
fig = px.scatter(sample, x='EstimatedSalary', y='Balance', color='Outcome',
                 size='NumOfProducts', opacity=.6, title='Balance vs estimated salary')
fig.show()

numeric = df.select_dtypes(include='number').drop(columns=['Year', 'CustomerId'], errors='ignore')
fig = px.imshow(numeric.corr(), text_auto='.2f', color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, title='Numeric feature correlation heatmap')
fig.show()

## Interpretation notes

- `Exited` is the target: 1 means churned, 0 means retained.
- `CustomerId` and `Surname` are identifiers and should not be model predictors.
- A constant field such as `Year` in this dataset carries no predictive information.
- Treat demographic and location comparisons as descriptive signals that need fairness review.
- Use the Streamlit app's held-out metrics for predictive evaluation; these EDA charts are not model performance results.